# 00 — Locations CSV first-pass inspection

Profiles the provided `data/raw/locations.csv` (~1M rows) with DuckDB — no full load into RAM.
Feeds the **data-quality issues** section of `docs/data-sources.md`.

Drop the provided file at `data/raw/locations.csv` before running (see `data/raw/README.md`).

In [ ]:
import duckdb
from leo_pipeline.config import PATHS

CSV = PATHS.locations_csv
assert CSV.exists(), f"Place the provided file at {CSV} (see data/raw/README.md)"
con = duckdb.connect()
rel = f"read_csv_auto('{CSV}', sample_size=-1)"  # lazy: read on demand, no full load into RAM
print(CSV)

In [ ]:
# Schema + row count
print(con.sql(f"DESCRIBE FROM {rel}"))
print(con.sql(f"SELECT count(location_id) AS n_rows FROM {rel}"))

In [ ]:
# Quality checks — adjust lat/lon column names to the real schema once known.
LAT, LON = 'latitude', 'longitude'
con.sql(f"""
SELECT
  count(*)                                                    AS total,
  count(*) FILTER (WHERE {LAT} IS NULL OR {LON} IS NULL)      AS null_coords,
  count(*) FILTER (WHERE {LAT} NOT BETWEEN -90 AND 90)        AS bad_lat,
  count(*) FILTER (WHERE {LON} NOT BETWEEN -180 AND 180)      AS bad_lon,
  count(*) FILTER (WHERE {LAT} = 0 AND {LON} = 0)             AS null_island
FROM {rel}
""")

In [ ]:
# Duplicate coordinates
con.sql(f"""
SELECT count(*) AS duplicate_coord_groups FROM (
  SELECT {LAT}, {LON} FROM {rel} GROUP BY 1, 2 HAVING count(*) > 1
)
""")

In [ ]:
# Bounding box of all points — are they all in the US?
# CONUS bbox: lon -124.8..-66.9, lat 24.4..49.4 | Full US incl. AK/HI: lat 18..71.5, lon -179.2..-66.9
bbox = con.sql(f"""
SELECT
  min({LAT}) AS min_lat, max({LAT}) AS max_lat,
  min({LON}) AS min_lon, max({LON}) AS max_lon,
  count(*)                                                                       AS total,
  count(*) FILTER (WHERE {LAT} BETWEEN 24.4 AND 49.4
                     AND {LON} BETWEEN -124.8 AND -66.9)                         AS in_conus,
  count(*) FILTER (WHERE {LAT} BETWEEN 18.0 AND 71.5
                     AND {LON} BETWEEN -179.2 AND -66.9)                         AS in_us_full,
  count(*) FILTER (WHERE NOT ({LAT} BETWEEN 18.0 AND 71.5
                     AND {LON} BETWEEN -179.2 AND -66.9))                        AS outside_us
FROM {rel}
WHERE {LAT} IS NOT NULL AND {LON} IS NOT NULL
""")
print(bbox)
row = bbox.fetchone()
print(f"\nBBox: lat [{row[0]}, {row[1]}], lon [{row[2]}, {row[3]}]")
print(f"All points within the US bbox? {row[7] == 0}  (outside_us = {row[7]} of {row[4]})")